# 2 - Training U-Net for Gap-Filling (Generic - Streaming)

**Self-supervised streaming training pipeline** for any Level-3 ocean variable.

This notebook trains a U-Net to fill gaps in satellite ocean observations using **xbatcher streaming** to keep memory bounded. Works for:
- Chlorophyll-a (PACE, Copernicus, CMEMS)
- Sea surface temperature
- Any other gridded ocean variable with cloud/missing data

## Key Features

- **Streaming workflow**: Zarr → xarray/Dask → xbatcher → TensorFlow
- **Memory-bounded**: Never loads full dataset into RAM
- **Spatial chunking**: 40×56 tiles (or configurable)
- **Generic target variable**: Not hardcoded to chlorophyll
- **Self-supervised**: No gap-free truth needed  
- **Synthetic clouds**: Temporally-correlated fake gaps for training/eval

## Workflow

```
Zarr (on-disk)
  ↓ xr.open_zarr with chunks
Lazy xarray/Dask Dataset
  ↓ build_standardized_lazy (no .load())
Lazy standardized channels
  ↓ xbatcher.BatchGenerator
Spatial tiles (time × 40×56)
  ↓ make_tf_gen → tf.data.Dataset
Training batches
  ↓
model.fit()
```

## Setup

In [ ]:
# Install mindthegap if needed
# %pip install --force-reinstall --no-cache-dir "git+https://github.com/SAFS-Varanasi-Internship/mindthegap.git@streamlined-pipeline"

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # Reduce TensorFlow verbosity

import numpy as np
import pandas as pd
import xarray as xr
import tensorflow as tf
import matplotlib.pyplot as plt
import mindthegap as mtg

# Enable GPU memory growth (prevents cuDNN crashes)
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {len(tf.config.list_physical_devices('GPU'))}")

## 1. Load Your Data

Load your xarray Dataset with **time, lat, lon** dimensions.

### Requirements:
- Target variable (e.g., `chlor_a`, `sst`, `analysed_sst`)
- Cloud/missing flag variable (1 = cloud/missing, 0 = valid data)
- Land flag variable (1 = land, 0 = ocean)
- Optional: Additional predictor variables (SST, winds, salinity, etc.)

### Data Source Examples:

In [ ]:
# Example 1: PACE Chlorophyll (AWS us-west-2 required)
# Uncomment to use:
"""
import earthaccess
import icechunk as ic

def load_pace_chl(lat_slice, lon_slice):
    auth = earthaccess.login()
    creds = auth.get_s3_credentials(daac="OBDAAC")
    
    url = "https://data.source.coop/fish-pace/pace-oci/inregion/PACE_OCI_L3M_CHL"
    storage = ic.http_storage(url)
    vc = ic.credentials.containers_credentials({
        "s3://ob-cumulus-prod-public/": ic.credentials.s3_credentials(
            access_key_id=creds["accessKeyId"],
            secret_access_key=creds["secretAccessKey"],
            session_token=creds["sessionToken"]
        )
    })
    
    store = ic.Repository.open(storage, authorize_virtual_chunk_access=vc).readonly_session("main").store
    ds = xr.open_zarr(store, consolidated=False, group="daily/0p1deg/chunks_512", chunks={})
    ds = ds.sel(lat=lat_slice, lon=lon_slice)
    
    # Derive land and cloud flags (PACE doesn't have explicit flags)
    gap = ds['chlor_a'].isnull()
    land = gap.all(dim='time')
    ds['land_flag'] = land.astype('int8')
    ds['cloud_flag'] = (gap & ~land).astype('int8')
    
    return ds

ds = load_pace_chl(lat_slice=slice(31, 5), lon_slice=slice(42, 80))
target_var = 'chlor_a'
missing_flag_var = 'cloud_flag'
land_flag_var = 'land_flag'
"""
pass

In [ ]:
# Example 2: Copernicus GlobColour (via icechunk)
import icechunk as ic

def open_globcolour(lat_slice, lon_slice):
    """Open the public GlobColour Icechunk store as xarray (no auth needed)."""
    url = "https://data.source.coop/fish-pace/globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D"
    storage = ic.http_storage(url)
    repo = ic.Repository.open(storage)
    containers = repo.config.virtual_chunk_containers or []
    store = ic.Repository.open(storage,
        authorize_virtual_chunk_access={p: ic.credentials.HttpAccess for p in containers}
    ).readonly_session("main").store
    return xr.open_zarr(store, consolidated=False, chunks={})

ds = open_globcolour(lat_slice=slice(31, 5), lon_slice=slice(42, 80))

target_var = 'CHL'
missing_flag_var = 'cloud_flag'
land_flag_var = 'land_flag'

In [ ]:
# Example 3: IO (Google Cloud Storage)
# Uncomment to use:
"""
ds = xr.open_dataset(
    "gcs://nmfs_odp_nwfsc/CB/mind_the_chl_gap/IO_rechunked.zarr",
    engine="zarr",
    backend_kwargs={"storage_options": {"token": "anon"}},
    consolidated=True
)
ds = ds.sel(lat=slice(31, 5), lon=slice(42, 80))

target_var = 'CHL_cmes-level3'
missing_flag_var = 'CHL_cmes-cloud'
land_flag_var = 'CHL_cmes-land'
"""
pass

In [ ]:
# =============================================================================
# USER CONFIGURATION: Modify this cell for your dataset
# =============================================================================

# Replace with your actual data loading code (use one of the examples above)
ds = xr.open_zarr("your_data.zarr")  # REPLACE THIS

# Specify your variable names
target_var = "chlor_a"  # Variable to gap-fill
missing_flag_var = "cloud_flag"  # 1 = missing/cloud, 0 = valid
land_flag_var = "land_flag"  # 1 = land, 0 = ocean

# Optional: Additional predictor features (empty list if none)
feature_vars = []  # e.g., ['sst', 'u_wind', 'v_wind', 'air_temp']

# Preprocessing options
log_transform = True  # Apply log to target? (True for chlorophyll, False for SST)
n_temporal_lags = 1  # Number of prev/next day channels (1 = prev1, next1)

# =============================================================================

print(f"Dataset dimensions: {dict(ds.sizes)}")
print(f"Target variable: {target_var}")
print(f"Date range: {pd.to_datetime(ds.time.values[0])} to {pd.to_datetime(ds.time.values[-1])}")

In [ ]:
# Crop to U-Net-compatible dimensions (multiples of 8)
ds = mtg.crop_to_multiple(ds, multiple=8)

print(f"\nAfter cropping to multiple of 8:")
print(f"Dimensions: {dict(ds.sizes)}")

## 2. Build Standardized Channels (Lazy/Streaming)

Use `build_standardized_lazy_new()` to create standardized predictors **without loading data into memory**.

### Streaming Configuration:
- `output_chunks`: Align Dask chunks with spatial tiles (e.g., `{"time": 100, "lat": 40, "lon": 56}`)
- Result is lazy xarray/Dask dataset—no `.load()` call
- xbatcher will read tiles on-demand during training

In [ ]:
# =============================================================================
# STREAMING CONFIGURATION for xbatcher
# =============================================================================

# Spatial tile size (must be divisible by 8 for U-Net)
TILE_LAT = 40
TILE_LON = 56

# Time chunk size (days per xbatcher block)
TIME_CHUNK = 100

# Training configuration
BATCH_SIZE = 16  # Mini-batch size for GPU
EPOCHS = 50
PATIENCE = 10

# REPLACE with RANDOM
# Train/val/test split (in years or as fractions)
train_start = str(pd.to_datetime(ds.time.values[0]).date())
train_years = 3  # Use 3 years for training stats
val_years = 1

train_end_date = pd.to_datetime(ds.time.values[0]) + pd.DateOffset(years=train_years)
val_end_date = train_end_date + pd.DateOffset(years=val_years)

print(f"Training period: {train_start} to {train_end_date.date()}")
print(f"Validation period: {train_end_date.date()} to {val_end_date.date()}")
print(f"Tile size: {TILE_LAT}×{TILE_LON}, Time chunks: {TIME_CHUNK} days")

In [ ]:
# Build lazy standardized dataset
print("Building lazy standardized channels...")

# Align output chunks with tile size for efficient xbatcher reads
output_chunks = {"time": TIME_CHUNK, "lat": TILE_LAT, "lon": TILE_LON}

ds_std, stats = mtg.build_standardized_lazy_new(
    ds,
    target_variable=target_var,
    missing_flag=missing_flag_var,
    land_flag=land_flag_var,
    features=feature_vars,
    train_dates=slice(train_start, str(train_end_date.date())),
    std_vars=feature_vars,  # Standardize features (target handled separately)
    log_target=log_transform,
    missing_flag_shift=10,
    n_temporal_lags=n_temporal_lags,
    output_chunks=output_chunks,
    add_geo=False  # Set True to add spherical lat/lon features
)

# Extract channel names (all except full_target)
X_vars = [v for v in ds_std.data_vars if v != 'full_target']
num_channels = len(X_vars)

# Get standardization stats
# Note: build_standardized_lazy_new uses 'full_target' but includes 'CHL' alias for compatibility
y_mean, y_std = stats['full_target'][0], stats['full_target'][1]

print(f"\nChannels created ({num_channels} total):")
for i, ch in enumerate(X_vars, 1):
    print(f"  {i}. {ch}")
print(f"\nTarget standardization: mean={y_mean:.4f}, std={y_std:.4f}")
print(f"\nDataset is LAZY (not in memory): {ds_std.chunks}")

## 3. Create xbatcher Streaming Pipeline

Use `mtg.make_xbatcher()` to create tile generators, then wrap with `mtg.make_tf_gen()` for TensorFlow.

In [ ]:
# Define spatial patch dimensions
patch_dims = {"time": TIME_CHUNK, "lat": TILE_LAT, "lon": TILE_LON}

# Split data into train/val subsets
ds_train = ds_std.sel(time=slice(train_start, str(train_end_date.date())))
ds_val = ds_std.sel(time=slice(str(train_end_date.date()), str(val_end_date.date())))

print(f"\nTrain time range: {ds_train.time.values[0]} to {ds_train.time.values[-1]}")
print(f"Val time range: {ds_val.time.values[0]} to {ds_val.time.values[-1]}")

# Verify chunk alignment (informational only)
print("\nChecking original chunk alignment...")
print(f"  On-disk chunks (lat): {ds_std.chunksizes.get('lat', 'N/A')}")
print(f"  On-disk chunks (lon): {ds_std.chunksizes.get('lon', 'N/A')}")
print(f"  Patch dims (lat): {patch_dims['lat']}")
print(f"  Patch dims (lon): {patch_dims['lon']}")
lat_chunks = ds_std.chunksizes.get('lat', [])
lon_chunks = ds_std.chunksizes.get('lon', [])
aligned = (lat_chunks and lat_chunks[0] == patch_dims['lat'] and 
           lon_chunks and lon_chunks[0] == patch_dims['lon'])
print(f"  Aligned: {aligned}")
print("  (Note: Already loaded into memory, so alignment only affects .load() speed)")

# Create xbatcher generators on IN-MEMORY data
print("\nCreating xbatcher generators on in-memory data...")
train_batcher = mtg.make_xbatcher(ds_train, patch_dims, overlap=None)
val_batcher = mtg.make_xbatcher(ds_val, patch_dims, overlap=None)

print(f"Train tiles: {len(train_batcher)}")
print(f"Val tiles: {len(val_batcher)}")

# Calculate steps per epoch
train_steps = (len(train_batcher) * TIME_CHUNK) // BATCH_SIZE
val_steps = (len(val_batcher) * TIME_CHUNK) // BATCH_SIZE
print(f"\nSteps per epoch:")
print(f"  Train: {train_steps}")
print(f"  Val: {val_steps}")

In [ ]:
# Wrap xbatcher generators with TensorFlow Dataset
print("Creating TensorFlow datasets...")

output_signature = (
    tf.TensorSpec(shape=(TILE_LAT, TILE_LON, num_channels), dtype=tf.float32),
    tf.TensorSpec(shape=(TILE_LAT, TILE_LON, 1), dtype=tf.float32),
)

# Note: make_tf_gen expects label variable name
# build_standardized_lazy_new uses 'full_target' as the target variable
train_dataset = tf.data.Dataset.from_generator(
    mtg.make_tf_gen(train_batcher, X_vars, label='full_target'),
    output_signature=output_signature
).shuffle(512).batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    mtg.make_tf_gen(val_batcher, X_vars, label='full_target'),
    output_signature=output_signature
).batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)

print("✓ TensorFlow datasets ready")
print(f"  Train: shuffle(512) → batch({BATCH_SIZE}) → repeat → prefetch")
print(f"  Val: batch({BATCH_SIZE}) → repeat → prefetch")

## 4. Build U-Net Model

Fully-convolutional U-Net that can accept any spatial size (trains on 40×56, can predict on full domain).

In [ ]:
# Build U-Net
model = mtg.UNet((None, None, num_channels))  # Fully-convolutional

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse'
)

model.summary()

print(f"\nModel input shape: (batch, {TILE_LAT}, {TILE_LON}, {num_channels})")
print(f"Model output shape: (batch, {TILE_LAT}, {TILE_LON}, 1)")

## 5. Train with Streaming Data

Train using xbatcher streaming—data is loaded one tile at a time, keeping memory bounded.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

print(f"Starting training...")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Steps per epoch: train={train_steps}, val={val_steps}")
print(f"  Early stopping patience: {PATIENCE}")
print(f"\n" + "="*60)

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    steps_per_epoch=train_steps,
    validation_data=val_dataset,
    validation_steps=val_steps,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

print(f"Best val_loss: {min(history.history['val_loss']):.6f}")
print(f"Final train_loss: {history.history['loss'][-1]:.6f}")

## 5. Evaluate (Self-Supervised)

Almost certainly broken.

Evaluate on held-out **synthetic clouds** only (pixels we intentionally hid).
Compare to **persistence baseline** (yesterday's value).

In [ ]:
# Define test period (after training and validation)
test_start = val_end_date + pd.DateOffset(days=2)  # 2-day buffer
test_end = pd.to_datetime(ds.time.values[-1])

print(f"Test period: {test_start.date()} to {test_end.date()}")

# Select test data from original lazy dataset
ds_test = ds_std.sel(time=slice(str(test_start.date()), str(test_end.date())))

# Load test set into memory for prediction
print("Loading test data...")
ds_test_loaded = ds_test.load()

# Stack input channels
X_test = np.stack([ds_test_loaded[ch].values for ch in X_vars], axis=-1).astype('float32')
print(f"Test data shape: {X_test.shape}")

In [ ]:
# Predict on test set
print("\nPredicting on test set...")
predictions_std = model.predict(X_test, batch_size=1, verbose=1)

# Unstandardize predictions
predictions = predictions_std[..., 0] * y_std + y_mean

# Get ground truth from original data (in log-space if log_transform=True)
truth_data = ds[target_var].sel(time=slice(str(test_start.date()), str(test_end.date()))).values
if log_transform:
    truth = np.log(np.where(truth_data > 0, truth_data, np.nan))
else:
    truth = truth_data

# Get synthetic cloud mask for test period
synth_cloud_test = ds_test_loaded['synthetic_missing_flag'].values

# Compute metrics only at synthetic clouds (held-out observed pixels)
mask_valid = (synth_cloud_test == 1) & np.isfinite(truth) & np.isfinite(predictions)
test_mae = np.mean(np.abs(predictions[mask_valid] - truth[mask_valid]))
test_rmse = np.sqrt(np.mean((predictions[mask_valid] - truth[mask_valid])**2))

print(f"\nTest set pixels evaluated: {mask_valid.sum():,}")
print(f"U-Net MAE:  {test_mae:.4f}")
print(f"U-Net RMSE: {test_rmse:.4f}")

In [ ]:
# Persistence baseline (yesterday's value)
print("\nComputing persistence baseline...")
if log_transform:
    all_truth = np.log(np.where(ds[target_var].values > 0, ds[target_var].values, np.nan))
else:
    all_truth = ds[target_var].values

# Get test period indices in full dataset
test_time_idx = np.where((ds.time.values >= np.datetime64(test_start)) & 
                         (ds.time.values <= np.datetime64(test_end)))[0]
persistence = all_truth[test_time_idx - 1]  # Previous day's value

mask_persist = (synth_cloud_test == 1) & np.isfinite(truth) & np.isfinite(persistence)
persist_mae = np.mean(np.abs(persistence[mask_persist] - truth[mask_persist]))
persist_rmse = np.sqrt(np.mean((persistence[mask_persist] - truth[mask_persist])**2))

print(f"Persistence baseline:")
print(f"  MAE:  {persist_mae:.4f}")
print(f"  RMSE: {persist_rmse:.4f}")

In [ ]:
# Summary
print("\n" + "="*60)
print("SELF-SUPERVISED TEST RESULTS")
print("="*60)
print(f"Target variable: {target_var}")
print(f"Evaluation metric: {'log-space' if log_transform else 'linear'} MAE/RMSE")
print(f"Evaluated at {mask_valid.sum()} synthetic cloud pixels")
print()
print(f"{'Metric':<15} {'U-Net':>10} {'Persist':>10} {'Improve':>10}")
print("-" * 60)
print(f"{'MAE':<15} {test_mae:>10.4f} {persist_mae:>10.4f} {(persist_mae-test_mae):>10.4f}")
print(f"{'RMSE':<15} {test_rmse:>10.4f} {persist_rmse:>10.4f} {(persist_rmse-test_rmse):>10.4f}")
print(f"{'% Better':<15} {(1-test_mae/persist_mae)*100:>9.1f}% {(1-test_rmse/persist_rmse)*100:>9.1f}%")
print("="*60)

if test_mae < persist_mae:
    print("\n✓ SUCCESS: Model beats persistence baseline!")
else:
    print("\n✗ Model does not beat persistence. Consider:")
    print("  - More training epochs or data")
    print("  - Additional predictor features (SST, winds, etc.)")
    print("  - Different missing_flag_shift value")
    print("  - Spatial patching (if memory-limited)")

## 6. Visualize Training History

This part is broken.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))

epochs_run = len(history.history['loss'])
ax.plot(history.history['loss'], label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title(f'Training History ({epochs_run} epochs)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nBest epoch: {np.argmin(history.history['val_loss']) + 1}")
print(f"Best val_loss: {min(history.history['val_loss']):.6f}")

## 7. Test Prediction (Full Domain)

Load one test frame and predict on the full domain to verify the model works.

In [ ]:
# Load one test day for full-domain prediction
test_date = str(val_end_date.date() + pd.Timedelta(days=30))  # 30 days after val
print(f"Test prediction date: {test_date}")

# Select and load test frame
ds_test = ds_std.sel(time=test_date).load()

# Stack channels
X_test = np.stack([ds_test[ch].values for ch in X_vars], axis=-1).astype('float32')
X_test = X_test[np.newaxis, ...]  # Add batch dimension

print(f"Test input shape: {X_test.shape}")

# Predict (fully-convolutional model handles any size)
y_pred = model.predict(X_test, verbose=0)[0, :, :, 0]

# Unstandardize
y_pred_orig = y_pred * y_std + y_mean

print(f"Prediction shape: {y_pred_orig.shape}")
print(f"Prediction range: [{np.nanmin(y_pred_orig):.4f}, {np.nanmax(y_pred_orig):.4f}]")
print("\n✓ Model successfully predicts on full domain")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Observed (with synthetic gaps)
observed = ds_test['masked_CHL'].values
axes[0].imshow(observed, cmap='viridis', origin='upper')
axes[0].set_title(f'Observed (with synthetic clouds)\\n{test_date}')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

# Prediction
im = axes[1].imshow(y_pred_orig, cmap='viridis', origin='upper')
axes[1].set_title(f'U-Net Prediction\\n{test_date}')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')

plt.colorbar(im, ax=axes, label='Log Chlorophyll-a', fraction=0.02)
plt.tight_layout()
plt.show()

## 8. Save Model

Save the trained model for later use.

In [ ]:
model_path = f'unet_generic_streaming_{train_years}yr.keras'
model.save(model_path)
print(f"✓ Model saved to: {model_path}")

# Save stats for later unstandardization
import pickle
stats_path = model_path.replace('.keras', '_stats.pkl')
with open(stats_path, 'wb') as f:
    pickle.dump(stats, f)
print(f"✓ Stats saved to: {stats_path}")